<a href="https://colab.research.google.com/github/beenishtech/fastapi-iris-flower-web-app/blob/main/Iris_classifier_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libraries

In [1]:
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Iris dataset load

In [2]:
# Load the built-in Iris dataset,(don't need to download any file)
iris = load_iris()

# Separate dataset into features (X) and labels (y)
X = iris.data
y = iris.target

print("Dataset loaded")

# Print the names of the features (columns)
print("Features naam:", iris.feature_names)

# Print the target class names (flower species)
print("flower's types (Target):", iris.target_names)

Dataset loaded
Features naam: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
flower's types (Target): ['setosa' 'versicolor' 'virginica']


#Train -Test split

In [3]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Random Forest Classifier model

In [4]:
# Initialize the Random Forest Classifier
model = RandomForestClassifier(random_state=42)

# Train the model on training data
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

#Evaluate Model accuracy

In [7]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate the accuracy score
accuracy = accuracy_score(y_test, y_pred)

# Print the final model accuracy in percentage
print(f"Model Accuracy : {accuracy * 100:.2f}%")

Model Accuracy : 100.00%


#Save model using joblib

In [6]:
# 4. Model ko 'iris_model.pkl' ke naam se save karein
joblib.dump(model, 'iris_model.pkl')
print("Model name 'iris_model.pkl' saved")

Model name 'iris_model.pkl' saved


# FastAPI Development

In [8]:
!pip install fastapi uvicorn pyngrok nest_asyncio # Install FastAPI, server, and async libraries
!pip install pyngrok            # Install pyngrok tunneling package

import os                       # Import system and OS utilities
from google.colab import userdata # Import Colab secrets manager
import threading                  # Import threading for parallel execution
import joblib                     # Import joblib to load the saved ML model
import nest_asyncio               # Import nest_asyncio for nested event loops
import uvicorn                    # Import uvicorn ASGI server to run FastAPI
from fastapi import FastAPI       # Import FastAPI core framework
from pydantic import BaseModel    # Import BaseModel for data validation
from pyngrok import ngrok         # Import ngrok for creating public URLs
from multiprocessing import Process # Import Process for background management

#Run async server in colab

In [9]:
nest_asyncio.apply()         # Fix Notebook errors for running the async server
print("FastAPI libraries install and sets") # Show success message that everything is ready



#if any server already running refresh it in background process
#if __name__ == "__main__":
  # server_process = Process(target=run_server)
    #server_process.start()
    #print("FastAPI Server in background port 8000  started")

FastAPI libraries install and sets


#APPLY API NGROk_TOKEN (with secret key)

In [10]:
ngrok_token = userdata.get('NGROK_TOKEN')   # Get the saved Ngrok token from Colab Secrets

os.environ["NGROK_TOKEN"] = ngrok_token     # Save token inside system environment
!ngrok config add-authtoken $NGROK_TOKEN    # Authenticate Ngrok using the saved token

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


# FastAPI Deployment & Model Prediction

###NOTE:- CLICK ON LINK THEN WRITE  /docs  IN THE END LINE OF URL IN SEARCH BAR THEN YOU WILL BE ABLE TO SEE DASHBOARD

###NOTE:- I MADE 3 TYPES OF DEPLOYMENT ,ONLY USE ONE AT A TIME.

In [19]:
app = FastAPI() # Create a basic FastAPI application instance

trained_model = joblib.load('iris_model.pkl') # Load the saved trained machine learning model

species_mapping = {0: "setosa", 1: "versicolor", 2: "virginica"} # Map numbers to flower names

class IrisInput(BaseModel): # Define the format for input data
    sepal_length: float     # Accept sepal length as a decimal number
    sepal_width: float      # Accept sepal width as a decimal number
    petal_length: float     # Accept petal length as a decimal number
    petal_width: float      # Accept petal width as a decimal number

@app.get("/")               # Create a GET route for the main page
def read_root():            # Function for the main page route
    return {"message": "Welcome to the Iris Classification API!."} # Return welcome message

@app.post("/predict")  # Create a POST route for getting predictions
def predict_species(data: IrisInput): # Function to process input and predict
    features = [[data.sepal_length, data.sepal_width, data.petal_length, data.petal_width]] # Group inputs into a list

    prediction_id = trained_model.predict(features)[0] # Get prediction ID from the model

    return {"prediction": species_mapping[prediction_id]} # Return the final flower name

def run_server():  # function to start the server
    uvicorn.run(app, host="127.0.0.1", port=8000) # Run FastAPI app on local port 8000

server_thread = threading.Thread(target=run_server) # Create a parallel thread for the server
server_thread.start()  # Start the server in the background

ngrok.kill()  # Close all previously running Ngrok links

public_url = ngrok.connect(8000)  # Create a new public link for port 8000
print("Public Link:", public_url)  # Show the live public URL

INFO:     Started server process [1579]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Public Link: NgrokTunnel: "https://renewal-kissing-eggshell.ngrok-free.dev" -> "http://localhost:8000"


#OR
#SECOND METHOD (DIRECT DASHBORD) JUST CLICK ON LINK

In [10]:
from fastapi.responses import HTMLResponse    # Import tool to return HTML pages if needed

app = FastAPI(docs_url="/") # Create FastAPI app and set main page as testing page

trained_model = joblib.load('iris_model.pkl') # Load the saved trained machine learning model
species_mapping = {0: "setosa", 1: "versicolor", 2: "virginica"} # Map numbers to flower names

class IrisInput(BaseModel): # Define the format for input data
    sepal_length: float  # Accept sepal length as a decimal number
    sepal_width: float   # Accept sepal width as a decimal number
    petal_length: float  # Accept petal length as a decimal number
    petal_width: float   # Accept petal width as a decimal number

@app.post("/predict")    # Create a POST route for getting predictions
def predict_species(data: IrisInput):  # Function to process input and predict

    features = [[data.sepal_length, data.sepal_width, data.petal_length, data.petal_width]] # Group inputs into a list

    prediction_id = trained_model.predict(features)[0] # Get prediction ID from the model

    return {"prediction": species_mapping[prediction_id]} # Return the final flower name


def run_server(): # function to start the server
    uvicorn.run(app, host="127.0.0.1", port=8000) # Run FastAPI app on local port 8000


server_thread = threading.Thread(target=run_server) # Create a parallel thread for the server
server_thread.start() # Start the server in the background


ngrok.kill() # Close all previously running Ngrok links
public_url = ngrok.connect(8000) # Create a new public link for port 8000


print("FINAL LIVE LINK:", public_url) # Print the main live URL generated by Ngrok
print(" DIRECT TESTING LINK DASHBOARD")  # Print a simple text label for the link dashboard

INFO:     Started server process [11935]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


FINAL LIVE LINK: NgrokTunnel: "https://renewal-kissing-eggshell.ngrok-free.dev" -> "http://localhost:8000"
 DIRECT TESTING LINK DASHBOARD


#OR
#THIRD METHOD USER-FRIENDLY (ANYONE CAN USE IT WITH EASE TO GET PREDICTIONS)

In [11]:
# import threading
# import joblib
# import nest_asyncio
# import uvicorn
# from fastapi import FastAPI
# from fastapi.responses import HTMLResponse
# from pydantic import BaseModel
# from pyngrok import ngrok

from fastapi.responses import HTMLResponse

# nest_asyncio.apply()
app = FastAPI()

# Model load karein
trained_model = joblib.load('iris_model.pkl')
species_mapping = {0: "setosa", 1: "versicolor", 2: "virginica"}

class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

# --- FRONTEND HTML PAGE (With Flower Images) ---
@app.get("/interface", response_class=HTMLResponse)
def frontend_page():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Iris Classifier with Images</title>
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <style>
            body { font-family: Arial, sans-serif; background-color: #f4f7f6; display: flex; justify-content: center; align-items: center; min-height: 90vh; margin: 0; }
            .container { background: white; padding: 30px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); max-width: 380px; width: 100%; }
            h2 { color: #2c3e50; text-align: center; margin-bottom: 5px; }
            p.sub { text-align: center; color: #7f8c8d; font-size: 13px; margin-bottom: 20px; }
            .form-group { margin-bottom: 15px; }
            label { display: block; margin-bottom: 5px; font-weight: bold; color: #34495e; font-size: 14px; }
            span.range { color: #e67e22; font-weight: normal; font-size: 12px; }
            input { width: 100%; padding: 10px; border: 1px solid #bdc3c7; border-radius: 6px; box-sizing: border-box; font-size: 16px; }
            button { width: 100%; padding: 12px; background-color: #18bc9c; color: white; border: none; border-radius: 6px; font-size: 16px; font-weight: bold; cursor: pointer; margin-top: 10px; }
            #result-box { margin-top: 20px; padding: 15px; border-radius: 6px; text-align: center; display: none; background-color: #d4edda; color: #155724; border: 1px solid #c3e6cb; }
            #result-text { font-size: 18px; font-weight: bold; margin-bottom: 10px; }
            #flower-img { width: 100%; max-height: 200px; border-radius: 8px; object-fit: cover; margin-top: 5px; display: none; border: 2px solid #18bc9c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h2>🌸 Iris Classifier</h2>
            <p class="sub">Enter values within the allowed standard ranges</p>

            <div class="form-group">
                <label>Sepal Length <span class="range">(Range: 4.3 - 7.9 cm)</span>:</label>
                <input type="number" id="sepal_length" step="0.1" value="6.1">
            </div>

            <div class="form-group">
                <label>Sepal Width <span class="range">(Range: 2.0 - 4.4 cm)</span>:</label>
                <input type="number" id="sepal_width" step="0.1" value="2.8">
            </div>

            <div class="form-group">
                <label>Petal Length <span class="range">(Range: 1.0 - 6.9 cm)</span>:</label>
                <input type="number" id="petal_length" step="0.1" value="4.7">
            </div>

            <div class="form-group">
                <label>Petal Width <span class="range">(Range: 0.1 - 2.5 cm)</span>:</label>
                <input type="number" id="petal_width" step="0.1" value="1.2">
            </div>

            <button onclick="makePrediction()">Predict Flower</button>

            <div id="result-box">
                <div id="result-text"></div>
                <img id="flower-img" src="" alt="Predicted Flower">
            </div>
        </div>

        <script>
            // Flower images database links
            const flowerImages = {
                'SETOSA': 'https://upload.wikimedia.org/wikipedia/commons/5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg',
                'VERSICOLOR': 'https://upload.wikimedia.org/wikipedia/commons/4/41/Iris_versicolor_3.jpg',
                'VIRGINICA': 'https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg'
            };

            async function makePrediction() {
                const data = {
                    sepal_length: parseFloat(document.getElementById('sepal_length').value),
                    sepal_width: parseFloat(document.getElementById('sepal_width').value),
                    petal_length: parseFloat(document.getElementById('petal_length').value),
                    petal_width: parseFloat(document.getElementById('petal_width').value)
                };

                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify(data)
                });

                const result = await response.json();
                const speciesName = result.prediction.toUpperCase();

                // Show result box
                document.getElementById('result-box').style.display = 'block';
                document.getElementById('result-text').innerHTML = '🌸 Result: ' + speciesName;

                // Show dynamic image
                const imgElement = document.getElementById('flower-img');
                imgElement.src = flowerImages[speciesName];
                imgElement.style.display = 'block';
            }
        </script>
    </body>
    </html>
    """

# --- BACKEND LOGIC ---
@app.post("/predict")
def predict_species(data: IrisInput):
    features = [[data.sepal_length, data.sepal_width, data.petal_length, data.petal_width]]
    prediction_id = trained_model.predict(features)[0]
    return {"prediction": species_mapping[prediction_id]}

def run_server():
    try:
        uvicorn.run(app, host="127.0.0.1", port=8000)
    except:
        pass

server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

# Clear old tunnels
try:
    ngrok.kill()
except:
    pass

tunnel = ngrok.connect(8000)
clean_url = tunnel.public_url

print("\n" + "="*60)
print("NEW IMAGE LINK:", clean_url + "/interface")
print("USER-FRIENDLY")
print("="*60)

INFO:     Started server process [22779]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



NEW IMAGE LINK: https://renewal-kissing-eggshell.ngrok-free.dev/interface
USER-FRIENDLY
